# 093 — Generación 3D y mundos sintéticos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución explicada

**Ejercicio 1.** α₁ = 1−e⁻¹ = 0.6321; α₂ = 1−e⁻⁰·⁵ = 0.3935; α₃ = 1−e⁻³ = 0.9502.
T₁ = 1; T₂ = e⁻¹ = 0.3679; T₃ = e⁻¹·⁵ = 0.2231. Pesos: w₁ = 0.6321;
w₂ = 0.3679·0.3935 = 0.1448; w₃ = 0.2231·0.9502 = 0.2120 (Σw = 0.9889 →
**1.1 % llega al fondo**). Color: C = 0.6321·0.2 + 0.1448·0.9 + 0.2120·0.5 =
**0.3627**. Nota: σ₃ = 3 es la densidad más alta pero w₃ < w₁ por la oclusión.

**Ejercicio 2.** Ambos **bajan**: al subir σ₁, la primera muestra absorbe más luz y
T₂ = e^(−σ₁δ₁) y T₃ = e^(−σ₁δ₁−σ₂δ₂) caen multiplicativamente. Verificación:
con σ₁ = 2, T₂ = e⁻² = 0.1353 → w₂ = 0.0533 (antes 0.1448); T₃ = e⁻²·⁵ = 0.0821 →
w₃ = 0.0780 (antes 0.2120). La transmitancia es un producto de exponenciales
(suma en el exponente): ocluir delante atenúa TODO lo de detrás.

**Ejercicio 3.** 3DGS: render rasterizado en tiempo real y recorte trivial (las
gaussianas son primitivas explícitas que se pueden podar por región), a costa de
cientos de MB de memoria — crítico en móvil, donde habría que podar/comprimir la
nube. NeRF: representación compacta (MB de pesos) pero render por integración de
rayos, inviable en tiempo real en gama media sin variantes aceleradas. Elección
razonable: **3DGS con poda/compresión agresiva**, aceptando pérdida de detalle.

**Ejercicio 4.** El contrato JSON expone `kind` y `evidence`; solo esa evidencia
inspeccionable autoriza conclusiones.


In [ ]:
result = run_lab("generation", seed=93)
assert result["kind"] == "generation"
assert result["evidence"]
show(result)


In [ ]:
# Verificación numérica de los ejercicios
import math

def render(sigmas, deltas, colores):
    T, acumulado, pesos = [], 0.0, []
    for s, d in zip(sigmas, deltas):
        T.append(math.exp(-acumulado))
        acumulado += s * d
    alphas = [1 - math.exp(-s * d) for s, d in zip(sigmas, deltas)]
    pesos = [t * a for t, a in zip(T, alphas)]
    C = sum(w * c for w, c in zip(pesos, colores))
    return alphas, T, pesos, C

# Ejercicio 1
alphas, T, pesos, C = render([1.0, 0.5, 3.0], [1.0]*3, [0.2, 0.9, 0.5])
print("alphas:", [f"{a:.4f}" for a in alphas])
print("T:     ", [f"{t:.4f}" for t in T])
print("pesos: ", [f"{w:.4f}" for w in pesos], f" suma={sum(pesos):.4f}")
print(f"C = {C:.4f}   luz al fondo = {1 - sum(pesos):.4f}")
assert abs(C - 0.3627) < 1e-3

# Ejercicio 2: sigma_1 sube a 2.0 -> w2 y w3 bajan
_, T2, pesos2, _ = render([2.0, 0.5, 3.0], [1.0]*3, [0.2, 0.9, 0.5])
print(f"con σ₁=2: w₂ {pesos[1]:.4f} → {pesos2[1]:.4f}, w₃ {pesos[2]:.4f} → {pesos2[2]:.4f}")
assert pesos2[1] < pesos[1] and pesos2[2] < pesos[2]

# Ejemplo del README: σ=(0.5, 1.0, 2.0), colores RGB por canal
for canal, cs in zip("RGB", [(1,0,0), (0,1,0), (0,0,1)]):
    *_, Cc = render([0.5, 1.0, 2.0], [1.0]*3, cs)
    print(f"canal {canal}: {Cc:.4f}")


## Reflexión

1. ¿Por qué NeRF hace que σ dependa solo de la posición x pero el color c dependa
   también de la dirección d, y qué efecto visual sería imposible si c ignorara d?
2. El problema Janus de DreamFusion (caras repetidas alrededor del objeto): ¿por qué
   es una consecuencia directa de usar un crítico 2D por vista, y qué información
   adicional lo mitigaría?
3. 3DGS entrena en minutos y renderiza a >100 fps donde NeRF tarda horas y segundos
   por frame: ¿qué se paga a cambio (memoria, edición, heurísticas) y en qué
   aplicación seguiría prefiriendo la representación implícita?
